# Iceberg Drift Prediction - Model Training

This notebook demonstrates model training and evaluation.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from iceberg_drift.data_processing import (
    download_iceberg_positions,
    download_era5_wind,
    download_copernicus_currents,
    match_environmental_data,
    engineer_features,
    create_targets,
    split_trajectories,
    get_feature_columns,
    create_dataloaders,
)
from iceberg_drift.data_processing.preprocessing import scale_features
from iceberg_drift.models import PINNDriftModel, create_model
from iceberg_drift.evaluation import TrainingConfig, train_model, validate_model, ValidationConfig
from iceberg_drift.utils import plot_trajectory_comparison

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Prepare Data (Quick Version)

In [ ]:
# Load or generate data
iceberg_df = download_iceberg_positions(
    output_dir="../data/raw",
    start_date="2020-01-01",
    end_date="2022-12-31",
    min_length_m=100,
    source="SYNTHETIC",
)

wind_ds = download_era5_wind(
    output_dir="../data/raw",
    start_date="2020-01-01",
    end_date="2022-12-31",
    bbox=(-180, -80, 180, -50),
)

current_ds = download_copernicus_currents(
    output_dir="../data/raw",
    start_date="2020-01-01",
    end_date="2022-12-31",
    bbox=(-180, -80, 180, -50),
    depth_levels=[0, 10, 50],
)

matched_df = match_environmental_data(iceberg_df, wind_ds, current_ds)
features_df = engineer_features(matched_df, add_cyclical_time=True, add_physics_features=True, add_lag_features=True)
targets_df = create_targets(features_df, prediction_horizon_hours=24, target_type="velocity")

train_df, val_df, test_df = split_trajectories(targets_df, random_seed=42)

feature_cols = get_feature_columns(train_df)
target_cols = [c for c in train_df.columns if c.startswith('target_')]

train_scaled, val_scaled, test_scaled, scaler = scale_features(
    train_df, val_df, test_df, feature_cols, scaler_type="standard"
)

# Create dataloaders
train_loader, val_loader, test_loader = create_dataloaders(
    train_scaled, val_scaled, test_scaled,
    feature_cols=feature_cols,
    target_cols=target_cols,
    sequence_length=4,  # 24 hours / 6 hours
    prediction_horizon=4,
    time_step_hours=6,
    batch_size=32,
    num_workers=0,  # 0 for notebook
    mode="sequence",
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Input dim: {len(feature_cols)}, Output dim: {len(target_cols)}")

## 2. Create and Train PINN Model

In [ ]:
# Create PINN model
model = PINNDriftModel(
    input_dim=len(feature_cols),
    output_dim=len(target_cols),
    hidden_dim=128,
    num_layers=2,
    dropout=0.2,
    prediction_horizon=4,
    ml_model_type="lstm",
    loss_weights={"data": 1.0, "physics": 0.5, "boundary": 0.1},
)

print(f"Model parameters: {model.get_num_params():,}")
print(model)

In [ ]:
# Training configuration
train_config = TrainingConfig(
    learning_rate=1e-3,
    epochs=30,
    batch_size=32,
    early_stopping_patience=10,
    output_dir="../output/checkpoints",
    metric_horizons=[6, 12, 24],
    use_amp=True,
)

# Train
trained_model, state = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_config,
    device="auto",
)

## 3. Evaluate on Test Set

In [ ]:
# Validate on test set
val_config = ValidationConfig(
    metric_horizons=[6, 12, 24],
    output_dir="../output/validation",
    plot_trajectories=True,
    plot_error_distribution=True,
    plot_skill_scores=True,
)

val_result = validate_model(
    trained_model,
    test_loader,
    config=val_config,
    device="auto",
)

## 4. Visualize Results

In [ ]:
# Print metrics
m = val_result.metrics
print(f"Position RMSE: {m.rmse_position_km:.2f} km")
print(f"Position MAE: {m.mean_position_error_km:.2f} km")
print(f"Direction Error: {m.mean_direction_error_deg:.1f}°")
print(f"Skill vs Persistence: {m.skill_score_vs_persistence:.3f}")
print(f"Skill vs Physics: {m.skill_score_vs_physics:.3f}")

print("\nPer-horizon metrics:")
for h, hm in m.horizon_metrics.items():
    print(f"  {h}h: RMSE={hm.get('rmse_position_km', 'N/A'):.1f} km, MAE={hm.get('mean_position_error_km', 'N/A'):.1f} km")

In [ ]:
# Plot a few trajectory comparisons
pred_u = val_result.predictions['u']
pred_v = val_result.predictions['v']
true_u = val_result.targets['u']
true_v = val_result.targets['v']
metadata = val_result.metadata

# Integrate velocities to positions for first few samples
from iceberg_drift.models.physics_model import integrate_trajectory_batch

n_plot = min(5, len(pred_u))
fig, axes = plt.subplots(1, n_plot, figsize=(5*n_plot, 10), subplot_kw={'projection': 'polar'})

for i in range(n_plot):
    meta = metadata[i]
    init_lat = meta.get('init_lat', -65)
    init_lon = meta.get('init_lon', 0)
    
    # Integrate
    pred_lats, pred_lons = integrate_trajectory_batch(
        np.array([init_lat]), np.array([init_lon]),
        pred_u[i:i+1], pred_v[i:i+1], dt=6*3600
    )
    true_lats, true_lons = integrate_trajectory_batch(
        np.array([init_lat]), np.array([init_lon]),
        true_u[i:i+1], true_v[i:i+1], dt=6*3600
    )
    
    ax = axes[i] if n_plot > 1 else axes
    ax.plot(pred_lons[0], pred_lats[0], 'r--', label='Predicted', transform=ccrs.PlateCarree())
    ax.plot(true_lons[0], true_lats[0], 'b-', label='True', transform=ccrs.PlateCarree())
    ax.plot(init_lon, init_lat, 'go', transform=ccrs.PlateCarree())
    ax.set_title(f"Sample {i}")
    ax.legend()

plt.tight_layout()
plt.show()